# Featherweight — Phase 6 Group C: serve (bnb-4bit + LoRA) + measure (Colab T4)

Measures throughput / p95 latency / $-per-1k for the retained **R0** model vs GPT-4o —
the cost/latency half of the thesis. Thin driver; logic is in `utils/cost.py`.

**Why bnb-4bit, not AWQ:** AWQ/GPTQ quantization has to load the merged **fp16** model
(~16 GB) to calibrate, which OOMs the free-tier T4 (16 GB VRAM, ~13 GB RAM) — confirmed at
runtime. So we serve the **pre-quantized bnb-4bit base + the R0 LoRA adapter** (the path
already validated in Phase 4): loads at ~5.5 GB, fits easily, and is still a legitimate
4-bit 8B served on a T4. AWQ would shave some latency but isn't feasible on free Colab;
`serve/merge_quantize.py` is retained for ≥24 GB hardware. Single session, no restart.

## 1. GPU + repo + install

In [ ]:
!nvidia-smi  # T4 (Turing, SM 75 -> fp16, not bf16)

In [ ]:
![ -d /content/Featherweight/.git ] || git clone https://github.com/Nishaant-Soni/Featherweight.git /content/Featherweight
%cd /content/Featherweight
# vLLM + the bnb-4bit serving deps (transformers<5 + bitsandbytes), per requirements-colab-serve.txt
%pip install -q -r requirements-colab-serve.txt
%pip install -q -e .
import sys
sys.path.insert(0, "/content/Featherweight/src")
from featherweight import config
print("ROOT_DIR:", config.ROOT_DIR, "| base:", config.BASE_MODEL_4BIT)

## 2. HF login + pull the R0 adapter

In [ ]:
from huggingface_hub import login, snapshot_download

login(token="your_huggingface_token_here")  # or use getpass() to input your token securely
ADAPTER_DIR = snapshot_download("your_username/featherweight-adapter")  # R0 (retained model)
print("adapter at:", ADAPTER_DIR)

## 3. Fixed prompt set
The same baseline held-out prompts used elsewhere; 200 for a stable throughput + p95.

In [ ]:
!python scripts/prep_data.py            # baseline data -> data/heldout.jsonl
from featherweight.eval import heldout
prompts = [r["prompt"] for r in heldout.load_heldout()[:200]]
print("prompts:", len(prompts))

## 4. Load bnb-4bit base + R0 LoRA in vLLM's offline `LLM`
Offline `LLM` (not the HTTP server — PRD §9 mitigation). Pre-quantized base loads at
~5.5 GB; `enable_lora` attaches R0. `--dtype half` (T4 has no bf16).

In [ ]:
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

llm = LLM(
    model=config.BASE_MODEL_4BIT,
    quantization="bitsandbytes",
    load_format="bitsandbytes",
    dtype="half",
    enable_lora=True,
    max_lora_rank=config.CONFIG.train.lora.r,   # 16
    max_model_len=config.CONFIG.eval.vllm_max_model_len,   # 8192
    gpu_memory_utilization=config.CONFIG.eval.vllm_gpu_memory_utilization,
    trust_remote_code=True,
)
sp = SamplingParams(temperature=0.0, max_tokens=512)
lora = LoRARequest("featherweight-ft", 1, ADAPTER_DIR)
print("vLLM loaded bnb-4bit base + R0 LoRA")

## 5. Measure throughput + latency
Throughput from one batched `generate`; per-request latency from a single-prompt loop
(vLLM batches the big call, so a real p50/p95/p99 needs one-at-a-time timing).

In [ ]:
import time

# throughput: one batched call over all prompts
t0 = time.time(); _ = llm.generate(prompts, sp, lora_request=lora); wall = time.time() - t0

# latency: time prompts one at a time (ms)
lat_ms = []
for p in prompts[:50]:
    t = time.time(); _ = llm.generate([p], sp, lora_request=lora); lat_ms.append((time.time() - t) * 1000)

print(f"batched {len(prompts)} prompts in {wall:.1f}s")

## 6. Compute + write `results/serving.md` (FT bnb-4bit+LoRA vs GPT-4o)

In [ ]:
from featherweight.utils import cost

stats = cost.latency_stats(lat_ms)
ft = {
    "throughput_req_s": cost.throughput(len(prompts), wall),
    "p50_ms": stats["p50"], "p95_ms": stats["p95"], "p99_ms": stats["p99"],
    "usd_per_1k": cost.cost_per_1k(cost.gpu_cost_usd(wall), len(prompts)),
}
csv_path, md_path = cost.write_serving({
    "featherweight-ft (bnb-4bit+LoRA)": ft,
    "gpt-4o-2024-11-20-FC": cost.gpt4o_serving_metrics(),
})
print(md_path.read_text())

## 7. Commit the result
`results/serving.{csv,md}` is written under the repo — paste the printed table back (or
download) to commit locally. This closes Phase 6 (cost/latency thesis).